# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [3]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    !cd ECE1508_GenAI && git pull

%cd ECE1508_GenAI

Cloning into 'ECE1508_GenAI'...
remote: Enumerating objects: 498, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 498 (delta 1), reused 3 (delta 0), pack-reused 488 (from 1)
Receiving objects: 100% (498/498), 18.60 MiB | 19.14 MiB/s, done.
Resolving deltas: 100% (202/202), done.
/content/ECE1508_GenAI


In [4]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 9.4 MB/s eta 0:00:00


In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [6]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI
plugins: typeguard-4.5.2, anyio-4.14.2, langsmith-0.10.2
collected 8 items                                                              

steven/tests/test_data_pipeline.py::test_reconstruct_prices_round_trip PASSED [ 12%]
steven/tests/test_data_pipeline.py::test_anchor_correction_matches_close_0_for_all_horizon_bars PASSED [ 25%]
steven/tests/test_data_pipeline.py::test_wick_components_non_negative PASSED [ 37%]
steven/tests/test_data_pipeline.py::test_build_window_shapes_and_masks PASSED [ 50%]
steven/tests/test_data_pipeline.py::test_to_patchtst_input_patch_padding_mask PASSED [ 62%]
steven/tests/test_data_pipeline.py::test_window_sampler_unique_and_within_bounds PASSED [ 75%]
steven/tests/test_data_pipeline.py::test_window_sampler_respects_split_boundary PASSED [ 87%]

## Train PatchTST (real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first instead of the full config.

In [7]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

18:51:08 device: cuda
18:51:09 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
18:51:09 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
18:51:16 epoch 1/20  train_loss=0.26290  val_loss=0.11262  (2.8s)
18:51:16   -> saved best checkpoint (val_loss=0.11262) to steven/outputs/patchtst_checkpoint.pt
18:51:18 epoch 2/20  train_loss=0.17937  val_loss=0.11444  (1.9s)
18:51:20 epoch 3/20  train_loss=0.16481  val_loss=0.10576  (2.0s)
18:51:20   -> saved best checkpoint (val_loss=0.10576) to steven/outputs/patchtst_checkpoint.pt
18:51:22 epoch 4/20  train_loss=0.14963  val_loss=0.10528  (1.9s)
18:51:22   -> saved best checkpoint (val_loss=0.10528) to steven/outputs/patchtst_checkpoint.pt
18:51:24 epoch 5/20  train_loss=0.14585  val_loss=0.09684  (1.9s)
18:51:24   -> saved best checkpoint (val_loss=0.09684) to steven/outputs/patchtst_checkpoint.pt
18:51:26 epoch 6/20  train_loss=0.14207  val_loss=0.09675  (1.8s)
18:51:26   -> saved best ch

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [8]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

18:51:55 device: cuda
18:51:55 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
18:51:55 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
18:51:59 epoch 1/30  beta=0.20  train_loss=0.74988 (kl=0.8072)  val_loss=0.38289 (kl=0.8022)  (2.4s)
18:51:59   -> saved best checkpoint (val_loss=0.38289) to steven/outputs/cvae_checkpoint.pt
18:52:01 epoch 2/30  beta=0.40  train_loss=0.64582 (kl=0.8046)  val_loss=0.52022 (kl=0.8000)  (1.4s)
18:52:02 epoch 3/30  beta=0.60  train_loss=0.77039 (kl=0.8000)  val_loss=0.67301 (kl=0.8000)  (1.4s)
18:52:03 epoch 4/30  beta=0.80  train_loss=0.90502 (kl=0.8004)  val_loss=0.82753 (kl=0.8000)  (1.4s)
18:52:05 epoch 5/30  beta=1.00  train_loss=1.01885 (kl=0.8001)  val_loss=0.94081 (kl=0.8000)  (1.4s)
18:52:06 epoch 6/30  beta=1.00  train_loss=0.98748 (kl=0.8000)  val_loss=0.92987 (kl=0.8000)  (1.4s)
18:52:08 epoch 7/30  beta=1.00  train_loss=0.98090 (kl=0.8000)  val_loss=0.92747 (kl=0.8000)  (1.4s)
18:52:09

## Evaluate both models on the fixed test set

In [9]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

18:52:44 device: cuda
18:52:44 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
18:52:44 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
18:52:44 evaluating on 3000 fixed test windows
18:52:45 wrote metrics to steven/outputs/metrics.json
18:52:45 overall: {
  "n_windows": 3000,
  "patchtst_reparam_mae_rmse": [
    0.10745159536600113,
    0.3753245770931244
  ],
  "cvae_reparam_mae_rmse": [
    0.18540965020656586,
    0.5076577067375183
  ],
  "patchtst_ohlc_mae_rmse": [
    11.363617754625002,
    14.94349856382411
  ],
  "cvae_ohlc_mae_rmse": [
    35.53539831106466,
    50.66234554382078
  ],
  "patchtst_volume_mae_rmse": [
    2002164.375,
    3511997.0
  ],
  "cvae_volume_mae_rmse": [
    3312041.5,
    4668141.0
  ],
  "patchtst_directional_accuracy": [
    0.484,
    0.48533333333333334,
    0.5033333333333333
  ],
  "cvae_directional_accuracy": [
    0.487,
    0.47,
    0.458
  ],
  "cvae_avg_sample_variance": [
    1301

## Pull results back down

Zips `steven/outputs/` (checkpoints, metrics.json, sample_plots) and downloads it -- or just `git add`/`commit`/`push` from here if you'd rather sync back through the repo.

In [10]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")

  adding: steven/outputs/ (stored 0%)
  adding: steven/outputs/sample_plots/ (stored 0%)
  adding: steven/outputs/sample_plots/long_start26495_ctx56.png (deflated 11%)
  adding: steven/outputs/sample_plots/short_start26707_ctx14.png (deflated 6%)
  adding: steven/outputs/sample_plots/long_start26193_ctx56.png (deflated 5%)
  adding: steven/outputs/sample_plots/medium_start24658_ctx35.png (deflated 10%)
  adding: steven/outputs/sample_plots/long_start26847_ctx56.png (deflated 9%)
  adding: steven/outputs/sample_plots/long_start25774_ctx56.png (deflated 6%)
  adding: steven/outputs/sample_plots/medium_start26039_ctx35.png (deflated 6%)
  adding: steven/outputs/sample_plots/short_start26791_ctx14.png (deflated 10%)
  adding: steven/outputs/sample_plots/short_start25539_ctx14.png (deflated 6%)
  adding: steven/outputs/sample_plots/medium_start26219_ctx35.png (deflated 10%)
  adding: steven/outputs/sample_plots/short_start25312_ctx14.png (deflated 11%)
  adding: steven/outputs/sample_plots/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>